In [3]:
###Document Structure

from langchain_core.documents import Document

In [5]:
doc = Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Krish Naik",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [6]:
## Create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [10]:
sample_texts={
"../data/text_files/python_intro.txt":"""Python Programming Introduction
Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use Extensive standard library
- Cross-platform compatibility Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

"../data/text_files/machine_learning.txt": """Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve 
from experience without being explicitly programmed. It focuses on developing computer programs that can 
access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems """
}


for filepath, content in sample_texts.items(): 
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)
      
print(" Sample text files created!")


 Sample text files created!


In [2]:
###Text loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()

print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use Extensive standard library\n- Cross-platform compatibility Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [10]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ##Pattern to match files
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)
documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve \nfrom experience without being explicitly programmed. It focuses on developing computer programs that can \naccess data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular program

In [ ]:
### Directory Loader

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the pdf files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ##Pattern to match files
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)
pdf_documents = dir_loader.load()

print(f"Loaded {len(pdf_documents)} pages")
print(pdf_documents[0].page_content)
print(pdf_documents[0].metadata)

In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(
    pdf_documents,
    chunk_size=500,
    chunk_overlap=100
)
chunks

### embedding And vectorStoreDB

In [32]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



### Manage Embedding

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
           model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""  
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts
            Args:
            texts: List of text strings to embed
            Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
           raise ValueError("Model not loaded")
       
        print (f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print (f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings        
    
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """             
            Initialize the vector store
            Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    
        
    def add_documents (self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to the vector store
            Args:
                documents: List of LangChain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
           raise ValueError("Number of documents must match number of embeddings")
       
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
        
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print (f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        

vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [43]:
chunks

[Document(metadata={'producer': 'Skia/PDF m150', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-07-29T14:44:08+00:00', 'source': '..\\data\\pdf\\Avirup_kar_CV.pdf', 'file_path': '..\\data\\pdf\\Avirup_kar_CV.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Avirup kar Resume', 'author': '', 'subject': '', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'moddate': '2026-07-29T14:44:08+00:00', 'trapped': '', 'modDate': "D:20260729144408+00'00'", 'creationDate': "D:20260729144408+00'00'", 'page': 0}, page_content='Avirup kar\navirupk57@gmail.com\n \nGitHub\n \nLinkedIn\n \nX.com\n \nPortfolio\n \nPROFILE\n-Full-stack developer building modern web applications with React and Next.js. I focus on clean UI, scalable \nbackend logic, and writing maintainable, performance-oriented code.\nI enjoy designing complete systems from frontend to backend while continuously improving my skills through \nreal-world projects.\nSKILLS\nFrontend & Backend Development